Config filenames in this copy are the names on main.

# Qian: BitFit cosine-head resolution ablation

Compare 336 vs 384 vs 448 on the same 1,000 classes, 40 fit + 5 holdout images/class, seed 42. Train all three fresh for three epochs with identical batch size and optimizer settings. Build separate fitting-only prototypes at each resolution. Select best.pt using crop 0.9 without flip; evaluate each checkpoint at crop 0.75 with and without flip. Record accuracy, memory and training throughput. Equal epochs compare data exposure; follow up with equal time if a higher resolution wins. GPU required; run all.


In [1]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "qq_bitfit_ablation"
TEAM = "08"
CONFIG = "configs/bitfit_resolution-sweep_all-biases_cosine-head_pilot.yaml"
EXPERIMENT_NAME = "qq_bitfit_resolution_336_384_448_seed42_v1"
EXPERIMENT_DIR = f"{DRIVE_OUT}/{EXPERIMENT_NAME}"
SPLIT_DIR = f"/content/{EXPERIMENT_NAME}_splits"
# Use a new name if changing the recipe or epoch count.
RESOLUTIONS = (336, 384, 448)


## 1. Clone or pull the repo

In [2]:
import base64
import os
import subprocess
from getpass import getpass

# Pass credentials through a temporary child-process environment, not the URL.
token = getpass("GitHub PAT (hidden; repo scope). Leave empty for a public repo: ")
git_env = os.environ.copy()
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraHeader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del auth
try:
    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], env=git_env, check=True)
        subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], env=git_env, check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], env=git_env, check=True)
finally:
    del token, git_env
os.chdir(REPO_DIR)
actual = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
assert actual == BRANCH, f"Wrong branch: {actual}"
print("Branch:", actual)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


GitHub PAT (hidden; repo scope). Leave empty for a public repo: ··········
Branch: qq_bitfit_ablation
Commit: 3ca3ff1


## 2. Install requirements

In [3]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [4]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

Mounted at /content/drive
Writing runs to /content/drive/.shortcut-targets-by-id/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR/CS604_RUNS
Existing items: ['full_ft_20261005_074553_id962394', 'full_ft_20261006_031830', 'headinit_full_ft_336_20261006_110343', 'prototypes_336.pt', 'prototypes_336.json', 'full_ft_20261006_122039', 'lora_20261007_051926', 'qq_selected_layer_4bae5d64c9c7ecfb', 'last_blocks_20261008_123344_id969549', 'qq_prefix_ablation_336_seed42_v1', 'pilot_adapter_336_20261009_125653', 'adapter_20261009_131150', 'prototypes_224.json', 'prototypes_224.pt', 'last_blocks_20261010_022115', 'val_team_Wenqi.txt', 'prefix_20261010_000404_id974631', 'qq_bitfit_ablation_336_subset_seed42_v2', 'prototypes_336.usage_metadata.json', 'bitfit_20261011_024724_id976355']


## 4. Download data to `/content/data`

In [5]:
subprocess.run(["python", "scripts/download_data.py", "--dest", DATA_DIR], check=True)


CompletedProcess(args=['python', 'scripts/download_data.py', '--dest', '/content/data'], returncode=0)

## 5. Train cosine-head BitFit at each resolution

Three epochs each, batch 64, bias/head LR 1e-3, zero weight decay, BF16, 5% warmup + cosine. All three arms use the same split and seed. Prototypes use fitting images only and are stored by resolution. If memory requires a smaller batch, change batch and accumulation for all three arms and use a new experiment name.


In [6]:
import torch
from src.utils import load_config, apply_overrides
from src.peft.bitfit_ablation import prepare, stage_config, ensure_prototypes, run_arm, write_results
import pandas as pd

assert torch.cuda.is_available(), "Select a CUDA GPU runtime."
BASE = load_config(CONFIG)
apply_overrides(BASE, data_root=DATA_DIR, output_dir=EXPERIMENT_DIR)
EXPERIMENT = Path(EXPERIMENT_DIR)
EXPERIMENT.mkdir(parents=True, exist_ok=True)
ROOTS = prepare(BASE, EXPERIMENT, split_dir=SPLIT_DIR)
results = []
for resolution in RESOLUTIONS:
    arm = {"arm": f"all_biases_cosine_{resolution}", "method": "bitfit", "bitfit_scope": "all", "head_type": "cosine", "img_size": resolution}
    cfg = stage_config(BASE, EXPERIMENT, ROOTS["pilot"], "resolution", arm, 42, 1e-3)
    ensure_prototypes(cfg, allow_missing_classes=True)
    result = run_arm(cfg)
    results.append({**result, "head_type": "cosine", "img_size": resolution})
write_results(EXPERIMENT, "resolution", results)
display(pd.DataFrame(results))
for result in results:
    subprocess.run(["python", "scripts/plot_curves.py", "--run_dir", result["run_dir"],
                    "--out_dir", str(Path(result["run_dir"]) / "figures")], check=True)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
(EXPERIMENT / "launcher_commit.txt").write_text(commit + "\n")


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Fit-only prototypes: 100%|██████████| 625/625 [01:07<00:00,  9.30it/s]


[prototypes] saved /content/drive/MyDrive/CS604_RUNS/qq_bitfit_resolution_336_384_448_seed42_v1/prototypes_pilot_336.pt


[bitfit] scope=all: unfroze 74 backbone bias tensors; classifier also trainable
[head] loaded prototypes from /content/drive/MyDrive/CS604_RUNS/qq_bitfit_resolution_336_384_448_seed42_v1/prototypes_pilot_336.pt  shape=(10000, 768)
[train] device=cuda method=bitfit model=vit_base_patch16_224.augreg2_in21k_ft_in1k
[train] run_dir=/content/drive/MyDrive/CS604_RUNS/qq_bitfit_resolution_336_384_448_seed42_v1/runs/resolution/all_biases_cosine_336/seed42/bitfit_20261014_131930
[bitfit] scope=all: unfroze 74 backbone bias tensors; classifier also trainable
[head] loaded prototypes from /content/drive/MyDrive/CS604_RUNS/qq_bitfit_resolution_336_384_448_seed42_v1/prototypes_pilot_336.pt  shape=(10000, 768)
[optim] backbone lr=0.001  head_lr=0.001
[train] images=40000, batches_per_epoch=625, batch_size=64, workers=8
[train] GPU=NVIDIA A100-SXM4-40GB
[train] stopping=epoch ceiling or time budget
[start] trainable=7.78M/93.67M (8.31%) wall_s=1.7 gpu_hours=0.0005 peak_vram_gb=1.199


epoch 1/3: 100%|██████████| 625/625 [01:20<00:00,  7.73it/s, loss=0.9835, lr=7.96e-04]


... (19 outputs left out)

epoch 3/3: 100%|██████████| 625/625 [02:25<00:00,  4.28it/s, loss=0.4027, lr=0.00e+00]


[epoch 3] train_loss=0.4616 val_loss=0.4262 val_acc=0.8976
[epoch 3] new best val_acc=0.8976
[end] trainable=7.78M/93.93M (8.29%) wall_s=476.2 gpu_hours=0.1323 peak_vram_gb=11.944
[train] done. checkpoints in /content/drive/MyDrive/CS604_RUNS/qq_bitfit_resolution_336_384_448_seed42_v1/runs/resolution/all_biases_cosine_448/seed42/bitfit_20261014_133259


,arm,seed,lr,bitfit_scope,before_val_acc,best_val_acc,best_epoch,final_val_acc,final_val_loss,early_dip,...,peak_vram_gb,training_minutes,wall_gpu_hours,training_images_per_second,gpu,torch_version,run_dir,timm_version,head_type,img_size
0,all_biases_cosine_336,42,0.001,all,0.6902,0.8888,3,0.8888,0.451067,False,...,7.011904,4.03,0.074538,496.668184,NVIDIA A100-SXM4-40GB,2.11.0+cu130,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,1.0.30,cosine,336
1,all_biases_cosine_384,42,0.001,all,0.6690,0.8916,3,0.8916,0.439794,False,...,8.941511,5.23,0.095765,382.762910,NVIDIA A100-SXM4-40GB,2.11.0+cu130,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,1.0.30,cosine,384
2,all_biases_cosine_448,42,0.001,all,0.6372,0.8976,3,0.8976,0.426152,False,...,11.944471,7.30,0.132287,274.016395,NVIDIA A100-SXM4-40GB,2.11.0+cu130,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,1.0.30,cosine,448


41

## 6. Holdout evaluation: crop 0.75 with/without flip

Also evaluate crop 0.9 without flip to reproduce checkpoint-selection accuracy. Results are development scores; one seed and three epochs do not establish convergence. No CodaBench submissions.


In [7]:
import gc
import math
from torch.utils.data import DataLoader
from src.data import ImageListDataset, build_transforms
from src.model import build_model
from src.predict import load_checkpoint
from src.utils import get_device, set_seed

@torch.no_grad()
def score_holdout(model, cfg, crop, flip, device):
    _, transform = build_transforms(cfg, eval_crop_pct=crop)
    dataset = ImageListDataset(cfg.data_root, "val", transform=transform)
    if not dataset.has_labels:
        raise ValueError("This sweep requires a labeled internal holdout.")
    loader = DataLoader(dataset, batch_size=int(cfg.batch_size), shuffle=False,
                        num_workers=int(cfg.num_workers), pin_memory=device.type == "cuda")
    correct, count, nll = 0, 0, 0.0
    for images, labels, _ in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16,
                            enabled=bool(cfg.bf16) and device.type == "cuda"):
            logits = model(images)
            mirrored = model(torch.flip(images, dims=[3])) if flip else None
        log_probs = logits.float().log_softmax(dim=1)
        if flip:
            log_probs = torch.logaddexp(log_probs, mirrored.float().log_softmax(dim=1)) - math.log(2)
        correct += int((log_probs.argmax(1) == labels).sum())
        count += labels.numel()
        nll += float(torch.nn.functional.nll_loss(log_probs, labels, reduction="sum"))
    return {"val_acc": correct / count, "val_nll": nll / count, "holdout_images": count}

rows = []
device = get_device()
for result in results:
    run_dir = Path(result["run_dir"])
    checkpoint = run_dir / "best.pt"
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    cfg = load_config(run_dir / "config.yaml")
    # Reattach current-session image symlinks; persisted manifests use the same split.
    cfg.data_root = str(ROOTS["pilot"])
    set_seed(int(cfg.seed))
    model = build_model(cfg).to(device)
    load_checkpoint(model, checkpoint, device)
    model.eval()
    for crop in (0.9, 0.75):
        for flip in ((False,) if crop == 0.9 else (False, True)):
            metrics = score_holdout(model, cfg, crop, flip, device)
            rows.append({"img_size": result["img_size"], "head_type": result["head_type"], "crop_pct": crop, "tta_flip": flip,
                         "best_epoch": result["best_epoch"], "checkpoint": str(checkpoint), **metrics})
            print(rows[-1])
            if crop == 0.9 and not flip and abs(metrics["val_acc"] - result["best_val_acc"]) > 0.001:
                raise RuntimeError("Baseline differs from recorded best accuracy; check the split/checkpoint.")
            pd.DataFrame(rows).to_csv(EXPERIMENT / "resolution_holdout_results.csv", index=False)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
table = pd.DataFrame(rows)
display(table.sort_values("val_acc", ascending=False))
display(table.pivot(index="crop_pct", columns=["img_size", "tta_flip"], values="val_acc"))


[bitfit] scope=all: unfroze 74 backbone bias tensors; classifier also trainable
[predict] missing keys (76): ['cls_token', 'pos_embed', 'patch_embed.proj.weight', 'blocks.0.norm1.weight', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.proj.weight', 'blocks.0.norm2.weight', 'blocks.0.mlp.fc1.weight']...
{'img_size': 336, 'head_type': 'cosine', 'crop_pct': 0.9, 'tta_flip': False, 'best_epoch': 3, 'checkpoint': '/content/drive/MyDrive/CS604_RUNS/qq_bitfit_resolution_336_384_448_seed42_v1/runs/resolution/all_biases_cosine_336/seed42/bitfit_20261014_131930/best.pt', 'val_acc': 0.8888, 'val_nll': 0.4510672523498535, 'holdout_images': 5000}
{'img_size': 336, 'head_type': 'cosine', 'crop_pct': 0.75, 'tta_flip': False, 'best_epoch': 3, 'checkpoint': '/content/drive/MyDrive/CS604_RUNS/qq_bitfit_resolution_336_384_448_seed42_v1/runs/resolution/all_biases_cosine_336/seed42/bitfit_20261014_131930/best.pt', 'val_acc': 0.896, 'val_nll': 0.4277696321487427, 'holdout_images': 5000}
{'img_size': 336, 'head_

,img_size,head_type,crop_pct,tta_flip,best_epoch,checkpoint,val_acc,val_nll,holdout_images
8,448,cosine,0.75,True,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8990,0.418256,5000
7,448,cosine,0.75,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8984,0.418888,5000
4,384,cosine,0.75,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8978,0.424400,5000
6,448,cosine,0.90,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8976,0.426152,5000
2,336,cosine,0.75,True,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8976,0.425058,5000
5,384,cosine,0.75,True,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8972,0.421516,5000
1,336,cosine,0.75,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8960,0.427770,5000
3,384,cosine,0.90,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8916,0.439794,5000
0,336,cosine,0.90,False,3,/content/drive/MyDrive/CS604_RUNS/qq_bitfit_re...,0.8888,0.451067,5000


img_size     336             384             448       
tta_flip   False   True    False   True    False  True 
crop_pct                                               
0.75      0.8960  0.8976  0.8978  0.8972  0.8984  0.899
0.90      0.8888     NaN  0.8916     NaN  0.8976    NaN